# RQ2 — Anteprima della separazione discovery/test

**Prima di usare:** eseguire e verificare in PyCharm `test_rq2_partition_preview.py`. Questo notebook stampa una **proposta** deterministica e verificata di distribuzione dei casi OOD: 4 query in discovery (una per classe, tutte da gruppi singoli), 11 query in test, con la finestra multi-fault indivisibile tutta nel test.

Il selettore legge ground truth **esclusivamente nel modulo di valutazione isolato**, non esporta etichette associate ai case_id. Nessun file scritto; **status DRAFT_NOT_EXECUTABLE**; nessuna chiamata API né autorizzazione a run. Confermare il protocollo con l'utente prima di un freeze.


In [ ]:
# RQ2 — SOLO ANTEPRIMA DELLA PARTIZIONE (non salva piano, zero API).
# Il modulo di valutazione legge il GT localmente per stratificare, ma
# stampa SOLO ID+fase, mai etichette per ID. Non importare in runner LLM.
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import sys

BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")
DATA = Path("/content/drive/MyDrive/Tesi_OpenRCA")
REPO_URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"
drive.mount("/content/drive")

for required in ("Market.zip",
                 "data/processed/split_market_v1.csv",
                 "data/processed/benchmark_inputs_market.csv"):
    if not (DATA / required).is_file():
        raise FileNotFoundError("Manca input congelato: " + required)

if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError("Directory OOD esistente non-Git: non sovrascrivo")
    current = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        text=True
    ).strip()
    if current != BRANCH:
        raise RuntimeError("Branch inatteso: non eseguo switch automatico")
    if subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        text=True
    ).strip():
        raise RuntimeError("Clone con modifiche locali: pull fermato")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True, text=True, capture_output=True
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch",
         "--branch", BRANCH, REPO_URL, str(REPO)],
        check=True, text=True, capture_output=True
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("=== RQ2 — ANTEPRIMA OFFLINE DELLA PARTIZIONE ===")
print("Branch:", BRANCH)
print("Commit:", commit)

env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")
run = subprocess.run(
    [sys.executable, "-m", "evaluation.rq2_partition_preview",
     "--data-dir", str(DATA)],
    cwd=str(REPO), env=env, text=True, capture_output=True
)
if run.returncode:
    raise RuntimeError("Anteprima non valida:\n" + run.stderr[-5000:])
try:
    data = json.loads(run.stdout)
except json.JSONDecodeError as exc:
    raise RuntimeError("Nessun JSON valido, anteprima NON verificata") from exc
if not (
    data.get("audit") == "rq2_stratified_partition_preview_v0"
    and data.get("status") == "DRAFT_NOT_EXECUTABLE"
    and data.get("source_verified") is True
    and data.get("discovery_query_count") == 4
    and data.get("test_query_count") == 11
    and data.get("discovery_window_groups") == 4
    and data.get("test_window_groups") == 10
    and data.get("read_ground_truth") is True
    and data.get("emits_case_label_mapping") is False
    and data.get("read_telemetry") is False
    and data.get("files_written") is False
    and data.get("model_api_calls") == 0
):
    raise RuntimeError("Vincoli del protocollo non verificati: nessuna approvazione")
if any(set(group) != {"phase", "case_ids"} for group in data["window_groups"]):
    raise RuntimeError("Campi inattesi nel report dei gruppi")
if (set(data["discovery_case_ids"]) & set(data["test_case_ids"])):
    raise RuntimeError("La stessa query compare in discovery e test")
print("\n=== PROPOSTA DI PARTIZIONE, NON PIANO CONGELATO ===")
print(json.dumps(data, indent=2, ensure_ascii=False))
print("\nNessuna inferenza, nessun GT per caso stampato, nessun file modificato.")


## Atteso

Al termine dovrebbero comparire 4 casi discovery, 11 test, 14 gruppi, 0 chiamate ai modelli. Non è una prova di indipendenza causale; l'analisi di generalizzazione resta limitata dal campione.
